In [25]:
# %%
# ============================================================
# 0. OpenAI API setup
# ============================================================

from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)

key = os.getenv("OPENAI_API_KEY")

if key:
    print("API key loaded successfully")
else:
    print("API key NOT found")

client = OpenAI()

MODEL = "gpt-5-mini"

API key loaded successfully


In [26]:
# %%
# ============================================================
# 1. Imports
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)

print("Packages loaded successfully.")

Packages loaded successfully.


In [27]:
# %%
# ============================================================
# 2. File paths
# ============================================================

BASE_DIR = Path(
    "/Users/yurujia/Desktop/Dissertation Data/China"
)

XINHUA_PATH = BASE_DIR / "excel/Xinhua_topic_multilabel_annotated_first_av_relevant_paragraph.xlsx"

OUTPUT_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_validation_results.xlsx"
)

print("Input file:")
print(XINHUA_PATH)

print("\nOutput file:")
print(OUTPUT_PATH)

Input file:
/Users/yurujia/Desktop/Dissertation Data/China/excel/Xinhua_topic_multilabel_annotated_first_av_relevant_paragraph.xlsx

Output file:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_validation_results.xlsx


In [28]:
# %%
# ============================================================
# 3. Load Xinhua validation sample
# ============================================================

xinhua = pd.read_excel(XINHUA_PATH)

print("Shape:", xinhua.shape)

print("\nColumns:")
print(xinhua.columns.tolist())

display(xinhua.head())

Shape: (137, 110)

Columns:
['sample_status', 'sample_has_av_relevant_paragraph', 'av_sentiment_auto', 'manual_sentiment', 'manual_sentiment_note', 'sentiment_analysis_text_preview', 'sentiment_analysis_text', 'first_av_relevant_paragraph_preview', 'first_av_relevant_paragraph', 'first_av_relevant_paragraph_before_dateline_removal', 'first_av_relevant_paragraph_found', 'first_av_relevant_paragraph_position', 'first_av_relevant_substantive_position', 'first_av_relevant_paragraph_char_count', 'first_av_relevant_paragraph_chinese_char_count', 'first_av_relevant_paragraph_english_word_count', 'first_av_relevant_paragraph_number_count', 'first_av_relevant_paragraph_approx_text_unit_count', 'av_relevant_xinhua_dateline_removed', 'av_paragraphs_checked_before_match', 'av_nonrelevant_substantive_paragraphs_skipped', 'av_skipped_nonrelevant_text', 'first_paragraph', 'first_paragraph_before_dateline_removal', 'first_paragraph_extracted', 'first_paragraph_chinese_char_count', 'first_paragraph_cha

,sample_status,sample_has_av_relevant_paragraph,av_sentiment_auto,manual_sentiment,manual_sentiment_note,sentiment_analysis_text_preview,sentiment_analysis_text,first_av_relevant_paragraph_preview,first_av_relevant_paragraph,first_av_relevant_paragraph_before_dateline_removal,...,topic_policy_regulation,topic_business_commercialisation,topic_public_acceptance_trust,topic_mobility_social_impact,topic_environment_sustainability,topic_legal_ethics,topic_other,topic_unclear,topic_primary,topic_label_count
0,retained_from_old_sample,1.0,1.0,1,NaN,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Business and Commercialisation,2.0
1,retained_from_old_sample,1.0,1.0,1,NaN,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Technology and Innovation,2.0
2,retained_from_old_sample,1.0,0.0,0,NaN,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,新华社天津６月２８日电（记者李鲲、钟群）无人驾驶的汽车如何应对现实环境中可能出现的各种问题？...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Technology and Innovation,2.0
3,retained_from_old_sample,1.0,1.0,1,NaN,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Business and Commercialisation,2.0
4,retained_from_old_sample,1.0,1.0,1,NaN,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Technology and Innovation,2.0


In [29]:
# %%
# ============================================================
# 4. Specify text and human-label columns
# ============================================================

TEXT_COL = "first_av_relevant_paragraph"
HUMAN_LABEL_COL = "manual_sentiment"

# Check whether columns exist
assert TEXT_COL in xinhua.columns, (
    f"{TEXT_COL} not found. "
    f"Available columns: {xinhua.columns.tolist()}"
)

assert HUMAN_LABEL_COL in xinhua.columns, (
    f"{HUMAN_LABEL_COL} not found. "
    f"Available columns: {xinhua.columns.tolist()}"
)

print("Text column:", TEXT_COL)
print("Human label column:", HUMAN_LABEL_COL)

Text column: first_av_relevant_paragraph
Human label column: manual_sentiment


In [30]:
# %%
# ============================================================
# 5. Standardise validation dataset
# ============================================================

validation_cn = pd.DataFrame({
    "source": "Xinhua",
    "text": xinhua[TEXT_COL],
    "human_sentiment": xinhua[HUMAN_LABEL_COL]
})

print("Original rows:", len(validation_cn))

# Remove missing text or labels
validation_cn = validation_cn.dropna(
    subset=["text", "human_sentiment"]
).copy()

# Clean text
validation_cn["text"] = (
    validation_cn["text"]
    .astype(str)
    .str.strip()
)

# Convert labels to integer
validation_cn["human_sentiment"] = (
    validation_cn["human_sentiment"]
    .astype(int)
)

# Keep only valid labels
validation_cn = validation_cn[
    validation_cn["human_sentiment"].isin([-1, 0, 1])
].reset_index(drop=True)

print("Usable rows:", len(validation_cn))

print("\nHuman label distribution:")
print(
    validation_cn["human_sentiment"]
    .value_counts()
    .sort_index()
)

display(validation_cn.head())

Original rows: 137
Usable rows: 116

Human label distribution:
human_sentiment
-1     2
 0    23
 1    91
Name: count, dtype: int64


,source,text,human_sentiment
0,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,1
1,Xinhua,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,1
2,Xinhua,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,0
3,Xinhua,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,1
4,Xinhua,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,1


In [31]:
# %%
# ============================================================
# 6. P1-CN: Basic Zero-Shot Prompt
# ============================================================

def build_prompt_p1_cn(text):
    return f"""
你正在为一项关于自动驾驶新闻报道的学术研究进行情感分类。

请判断以下新闻文本对自动驾驶汽车、自动驾驶技术及其发展与应用所表达的情感倾向。

请严格使用以下其中一个标签：

1 = 正面
0 = 中性
-1 = 负面

仅返回数值标签：1、0 或 -1。

新闻文本：
{text}
""".strip()

In [32]:
# %%
# ============================================================
# 7. P2-CN: Definition-Based Zero-Shot Prompt
# ============================================================

def build_prompt_p2_cn(text):
    return f"""
你正在为一项关于自动驾驶新闻报道的学术研究进行情感分类。

你的任务是判断以下新闻文本对自动驾驶汽车、自动驾驶技术及其发展与应用所表达的情感倾向。

请使用以下定义：

1 = 正面

文本整体以较为积极、有利的方式呈现自动驾驶汽车或自动驾驶技术。
这可能包括强调技术进步、创新、社会或经济效益、安全性提升、
成功测试或部署、商业扩张、支持性发展，或对未来发展的积极预期。

0 = 中性

文本整体以事实性、描述性、平衡性或混合性的方式报道自动驾驶，
没有明确表现出占主导地位的正面或负面评价。
文本可能报道技术进展、企业公告、政策、事件或不同观点，
但不存在明显的总体评价方向。

-1 = 负面

文本整体以较为消极、不利的方式呈现自动驾驶汽车或自动驾驶技术。
这可能包括强调技术失败、安全风险、事故、不可靠性、批评、
法律或监管问题、公众担忧、发展受挫或技术局限。

请判断的是文本对自动驾驶本身的情感倾向，
而不是新闻事件整体的一般情绪色彩。

仅返回数值标签：1、0 或 -1。

新闻文本：
{text}
""".strip()

In [33]:
# %%
# ============================================================
# 8. P3-CN: Rule-Guided Zero-Shot Prompt
# ============================================================

def build_prompt_p3_cn(text):
    return f"""
你正在为一项关于自动驾驶新闻报道的学术研究进行情感分类。

你的任务是判断以下文本对自动驾驶汽车、自动驾驶技术及其发展与应用的总体评价倾向。

请使用以下标签：

1 = 正面

文本整体以积极、有利的方式呈现自动驾驶，
包括技术进步、创新、社会或经济效益、安全性提升、
成功测试或部署、商业扩张、支持性政策发展，
或对未来发展的积极预期。

0 = 中性

文本整体以事实性、描述性、平衡性或混合性的方式呈现自动驾驶，
不存在明显占主导地位的正面或负面评价。

-1 = 负面

文本整体以消极、不利的方式呈现自动驾驶，
包括技术失败、安全风险、由自动驾驶相关问题引发的事故、
不可靠性、批评、监管或法律障碍、公众担忧、
发展受挫或技术局限。

请按照以下规则进行判断：

1. 只判断文本对自动驾驶汽车或自动驾驶技术的评价倾向，
不要依据新闻事件本身的一般情绪色彩进行判断。

2. 不要仅仅因为文本出现事故、受伤、死亡、调查、诉讼或监管等内容
就判断为负面。只有当这些内容对自动驾驶技术形成明显不利评价
或负面框架时，才应判断为负面。

3. 不要仅仅因为文本提到技术发展、投资、测试、产品发布、
服务上线或企业公告就判断为正面。
必须存在明确的积极评价、积极暗示或有利框架。

4. 对事件、政策公告、商业交易、测试、上线、扩张或监管变化的
纯事实性报道，通常应判断为中性，
除非文本明确以正面或负面的方式评价该发展。

5. 当文本同时包含正面和负面内容时，
判断哪一种评价方向占主导。
如果两者均不明显占主导，则判断为中性。

6. 对企业、公司高管、股价、财务表现或整体商业状况的情感，
不应直接决定自动驾驶情感标签，
除非这些内容明确反映了对自动驾驶技术或其部署的评价。

7. 引语中的情感只有在其明显影响整段文本对自动驾驶的总体呈现方式时，
才应纳入判断。

8. 只能依据所提供的文本作出判断。
不要使用关于企业、事件、事故、技术或政策的外部知识。

仅返回数值标签：1、0 或 -1。

新闻文本：
{text}
""".strip()

In [34]:
# %%
# ============================================================
# 9. API classification function
# ============================================================

def classify_sentiment(prompt, max_retries=3):

    for attempt in range(max_retries):

        try:
            response = client.responses.create(
                model=MODEL,
                input=prompt
            )

            result = response.output_text.strip()

            # Normal expected outputs
            if result in {"1", "0", "-1"}:
                return int(result)

            # Try to recover slightly malformed outputs
            cleaned = (
                result
                .replace("标签：", "")
                .replace("标签:", "")
                .replace("Label:", "")
                .replace(" ", "")
                .strip()
            )

            if cleaned in {"1", "0", "-1"}:
                return int(cleaned)

            print(
                f"Unexpected output: {repr(result)}"
            )

            return None

        except Exception as e:

            print(
                f"API error on attempt "
                f"{attempt + 1}/{max_retries}: {e}"
            )

            if attempt < max_retries - 1:
                time.sleep(3)

            else:
                return None

In [35]:
# %%
# ============================================================
# 10. Small test on first 5 articles
# ============================================================

test_cn = validation_cn.head(5).copy()

for idx, row in test_cn.iterrows():

    text = row["text"]

    p1 = classify_sentiment(
        build_prompt_p1_cn(text)
    )

    p2 = classify_sentiment(
        build_prompt_p2_cn(text)
    )

    p3 = classify_sentiment(
        build_prompt_p3_cn(text)
    )

    print("\n" + "=" * 80)
    print("Article:", idx + 1)
    print("Human:", row["human_sentiment"])
    print("P1-CN:", p1)
    print("P2-CN:", p2)
    print("P3-CN:", p3)

    print("\nText:")
    print(text[:300])


Article: 1
Human: 1
P1-CN: 1
P2-CN: 1
P3-CN: 1

Text:
王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶领域签署了多项合作文件，这是两个制造业大国优势互补、互利双赢的强强联合。他还说，中德双方深化务实合作，全方位、宽领域、高水平的特点更加鲜明，双方的合作形式越来越丰富，利益融合越来越深厚。

Article: 2
Human: 1
P1-CN: 1
P2-CN: 1
P3-CN: 0

Text:
此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造型中心哥德堡工作室、领克欧洲销售和市场团队等。建成后的创新中心面积约７００００－８００００平米，按照高艺术水准、高环保标准建造，主动安全、自动驾驶、互联互通等领域将成为其研究重点。

Article: 3
Human: 0
P1-CN: 1
P2-CN: 1
P3-CN: 1

Text:
无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，一场代表中国领先水平的智能驾驶比赛拉开帷幕。

Article: 4
Human: 1
P1-CN: 1
P2-CN: 1
P3-CN: 1

Text:
智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有人谈及智能出行和无人驾驶，而今天，已有许多中国企业希望与美方在这一重要的未来产业开展合作，“这是一个可喜变化”。

Article: 5
Human: 1
P1-CN: 1
P2-CN: 1
P3-CN: 1

Text:
自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，并在此次法兰克福车展上展示最新研发成果。


In [36]:
# %%
# ============================================================
# 11. Run P1-CN / P2-CN / P3-CN
# ============================================================

results_cn = validation_cn.copy()

results_cn["P1_CN"] = None
results_cn["P2_CN"] = None
results_cn["P3_CN"] = None


for idx, row in results_cn.iterrows():

    text = row["text"]

    print(
        f"Processing {idx + 1}/{len(results_cn)}"
    )

    results_cn.at[idx, "P1_CN"] = classify_sentiment(
        build_prompt_p1_cn(text)
    )

    results_cn.at[idx, "P2_CN"] = classify_sentiment(
        build_prompt_p2_cn(text)
    )

    results_cn.at[idx, "P3_CN"] = classify_sentiment(
        build_prompt_p3_cn(text)
    )

Processing 1/116
Processing 2/116
Processing 3/116
Processing 4/116
Processing 5/116
Processing 6/116
Processing 7/116
Processing 8/116
Processing 9/116
Processing 10/116
Processing 11/116
Processing 12/116
Processing 13/116
Processing 14/116
Processing 15/116
Processing 16/116
Processing 17/116
Processing 18/116
Processing 19/116
Processing 20/116
Processing 21/116
Processing 22/116
Processing 23/116
Processing 24/116
Processing 25/116
Processing 26/116
Processing 27/116
Processing 28/116
Processing 29/116
Processing 30/116
Processing 31/116
Processing 32/116
Processing 33/116
Processing 34/116
Processing 35/116
Processing 36/116
Processing 37/116
Processing 38/116
Processing 39/116
Processing 40/116
Processing 41/116
Processing 42/116
Processing 43/116
Processing 44/116
Processing 45/116
Processing 46/116
Processing 47/116
Processing 48/116
Processing 49/116
Processing 50/116
Processing 51/116
Processing 52/116
Processing 53/116
Processing 54/116
Processing 55/116
Processing 56/116
P

In [37]:
# %%
# ============================================================
# 12. Save raw results
# ============================================================

results_cn.to_excel(
    OUTPUT_PATH,
    index=False
)

print("Results saved successfully:")
print(OUTPUT_PATH)

Results saved successfully:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_validation_results.xlsx


In [38]:
# %%
# ============================================================
# 13. Check missing API predictions
# ============================================================

print(
    results_cn[
        ["P1_CN", "P2_CN", "P3_CN"]
    ].isna().sum()
)

P1_CN    0
P2_CN    0
P3_CN    0
dtype: int64


In [39]:
# %%
# ============================================================
# 14. Overall prompt evaluation
# ============================================================

prompt_columns_cn = [
    "P1_CN",
    "P2_CN",
    "P3_CN"
]

evaluation_rows_cn = []

for prompt_col in prompt_columns_cn:

    valid = results_cn[
        results_cn[prompt_col].notna()
    ].copy()

    y_true = valid[
        "human_sentiment"
    ].astype(int)

    y_pred = valid[
        prompt_col
    ].astype(int)

    evaluation_rows_cn.append({

        "prompt": prompt_col,

        "n": len(valid),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "cohen_kappa": cohen_kappa_score(
            y_true,
            y_pred
        )
    })


evaluation_cn_df = pd.DataFrame(
    evaluation_rows_cn
)

display(evaluation_cn_df)

,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P1_CN,116,0.836207,0.829133,0.637442,0.702502,0.476857
1,P2_CN,116,0.836207,0.828070,0.648272,0.709344,0.493217
2,P3_CN,116,0.775862,0.785661,0.655120,0.685957,0.423658


In [40]:
# %%
# ============================================================
# 15. Save overall evaluation metrics
# ============================================================

EVALUATION_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_evaluation_metrics.xlsx"
)

evaluation_cn_df.to_excel(
    EVALUATION_PATH,
    index=False
)

print("Saved:")
print(EVALUATION_PATH)

Saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_evaluation_metrics.xlsx


In [41]:
# %%
# ============================================================
# 16. Class-level performance
# ============================================================

for prompt_col in prompt_columns_cn:

    valid = results_cn[
        results_cn[prompt_col].notna()
    ].copy()

    y_true = valid[
        "human_sentiment"
    ].astype(int)

    y_pred = valid[
        prompt_col
    ].astype(int)

    print("\n" + "=" * 100)
    print(f"PROMPT: {prompt_col}")
    print("=" * 100)

    report = classification_report(
        y_true,
        y_pred,
        labels=[-1, 0, 1],
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(
        report
    ).T

    display(
        report_df[
            [
                "precision",
                "recall",
                "f1-score",
                "support"
            ]
        ]
    )


PROMPT: P1_CN


,precision,recall,f1-score,support
Negative,1.000000,0.500000,0.666667,2.000000
Neutral,0.611111,0.478261,0.536585,23.000000
Positive,0.876289,0.934066,0.904255,91.000000
accuracy,0.836207,0.836207,0.836207,0.836207
macro avg,0.829133,0.637442,0.702502,116.000000
weighted avg,0.825843,0.836207,0.827259,116.000000



PROMPT: P2_CN


,precision,recall,f1-score,support
Negative,1.000000,0.500000,0.666667,2.000000
Neutral,0.600000,0.521739,0.558140,23.000000
Positive,0.884211,0.923077,0.903226,91.000000
accuracy,0.836207,0.836207,0.836207,0.836207
macro avg,0.828070,0.648272,0.709344,116.000000
weighted avg,0.829855,0.836207,0.830725,116.000000



PROMPT: P3_CN


,precision,recall,f1-score,support
Negative,1.000000,0.500000,0.666667,2.000000
Neutral,0.454545,0.652174,0.535714,23.000000
Positive,0.902439,0.813187,0.855491,91.000000
accuracy,0.775862,0.775862,0.775862,0.775862
macro avg,0.785661,0.655120,0.685957,116.000000
weighted avg,0.815315,0.775862,0.788832,116.000000


In [42]:
# %%
# ============================================================
# 17. Compare label distributions
# ============================================================

columns_to_check_cn = [
    "human_sentiment",
    "P1_CN",
    "P2_CN",
    "P3_CN"
]

distribution_cn = {}

for col in columns_to_check_cn:

    counts = (
        results_cn[col]
        .dropna()
        .astype(int)
        .value_counts()
        .reindex(
            [-1, 0, 1],
            fill_value=0
        )
    )

    distribution_cn[col] = counts


distribution_cn_df = pd.DataFrame(
    distribution_cn
)

distribution_cn_df.index = [
    "Negative (-1)",
    "Neutral (0)",
    "Positive (1)"
]

display(distribution_cn_df)

,human_sentiment,P1_CN,P2_CN,P3_CN
Negative (-1),2,1,1,1
Neutral (0),23,18,20,33
Positive (1),91,97,95,82


In [43]:
# %%
# ============================================================
# 18. Compare label percentages
# ============================================================

distribution_pct_cn_df = (
    distribution_cn_df
    / distribution_cn_df.sum(axis=0)
    * 100
)

display(
    distribution_pct_cn_df.round(2)
)

,human_sentiment,P1_CN,P2_CN,P3_CN
Negative (-1),1.72,0.86,0.86,0.86
Neutral (0),19.83,15.52,17.24,28.45
Positive (1),78.45,83.62,81.90,70.69


In [44]:
# %%
# ============================================================
# 19. Confusion matrices
# ============================================================

for prompt_col in prompt_columns_cn:

    valid = results_cn[
        results_cn[prompt_col].notna()
    ].copy()

    y_true = valid[
        "human_sentiment"
    ].astype(int)

    y_pred = valid[
        prompt_col
    ].astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[-1, 0, 1]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            "Human Negative",
            "Human Neutral",
            "Human Positive"
        ],
        columns=[
            "Pred Negative",
            "Pred Neutral",
            "Pred Positive"
        ]
    )

    print("\n" + "=" * 80)
    print(prompt_col)
    print("=" * 80)

    display(cm_df)


P1_CN


,Pred Negative,Pred Neutral,Pred Positive
Human Negative,1,1,0
Human Neutral,0,11,12
Human Positive,0,6,85



P2_CN


,Pred Negative,Pred Neutral,Pred Positive
Human Negative,1,1,0
Human Neutral,0,12,11
Human Positive,0,7,84



P3_CN


,Pred Negative,Pred Neutral,Pred Positive
Human Negative,1,1,0
Human Neutral,0,15,8
Human Positive,0,17,74


In [45]:
# %%
# ============================================================
# 20. Error analysis
# ============================================================

for prompt_col in prompt_columns_cn:

    errors = results_cn[
        results_cn[prompt_col].astype(float)
        !=
        results_cn["human_sentiment"].astype(float)
    ].copy()

    print("\n" + "=" * 100)
    print(f"ERRORS FOR {prompt_col}")
    print("=" * 100)

    print(
        "Number of errors:",
        len(errors)
    )

    display(
        errors[
            [
                "text",
                "human_sentiment",
                prompt_col
            ]
        ]
    )


ERRORS FOR P1_CN
Number of errors: 19


,text,human_sentiment,P1_CN
2,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,0,1
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,1,0
16,“无人驾驶”拖拉机能够开进田间地头，源于中国正在应用的自主发展、独立运行的卫星导航系统——北...,0,1
17,据湖南磁浮公司董事长周晓明介绍，两年来该公司建立和完善了中低速磁浮列车的系统设计、制造、试验...,0,1
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,1,0
27,在北美轿车市场萎缩的情况下，通用汽车不久前宣布在美国和加拿大大幅减员，同时削减滞销车型的生产...,0,1
36,2022年杭州亚运会召开时，亚运区域内将全面实现自动驾驶。这是记者从13日第19届亚运会汽车...,1,0
37,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,-1,0
38,设计时速200公里的磁浮列车也在紧张研制当中，计划2020年初在中车株机公司下线。这款无人驾...,0,1
39,“新能源汽车跨界融合新趋势”是本次大会的主要议题之一。大会设置中重型车零排放论坛、城市交通电...,0,1



ERRORS FOR P2_CN
Number of errors: 19


,text,human_sentiment,P2_CN
10,英国目前正借助工业界和学术界的力量推动多项创新技术在国防领域的应用，此前已和美国合作，实地测...,0,1
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,1,0
16,“无人驾驶”拖拉机能够开进田间地头，源于中国正在应用的自主发展、独立运行的卫星导航系统——北...,0,1
17,据湖南磁浮公司董事长周晓明介绍，两年来该公司建立和完善了中低速磁浮列车的系统设计、制造、试验...,0,1
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,1,0
23,发布的15项世界互联网领先科技成果包括：微信小程序商业模式创新、华为昇腾310芯片、蚂蚁金服...,1,0
30,美德两大汽车制造商福特和大众15日宣布多项合作协议，将联合生产皮卡和厢式货车，并探索在电动汽...,1,0
37,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,-1,0
38,设计时速200公里的磁浮列车也在紧张研制当中，计划2020年初在中车株机公司下线。这款无人驾...,0,1
43,2016年8月，“费多尔”机器人原型机定型，身高180厘米，体重达160公斤，工作时功率约为...,1,0



ERRORS FOR P3_CN
Number of errors: 26


,text,human_sentiment,P3_CN
2,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,0,1
6,未来，芯片可被广泛应用于车辆管理、汽车导航、可穿戴设备、航海导航、ＧＩＳ数据采集、精准农业、...,1,0
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,1,0
17,据湖南磁浮公司董事长周晓明介绍，两年来该公司建立和完善了中低速磁浮列车的系统设计、制造、试验...,0,1
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,1,0
21,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,1,0
25,在自动驾驶领域，大众汽车集团正在中国加速发展自动驾驶技术。目前，奥迪品牌已在北京和无锡两座城...,1,0
30,美德两大汽车制造商福特和大众15日宣布多项合作协议，将联合生产皮卡和厢式货车，并探索在电动汽...,1,0
32,产业通商资源部表示，韩国吸引外国直接投资的重点领域将包括汽车自动驾驶、医疗、智慧家庭、节能方...,1,0
33,内塔尼亚胡在与斯洛伐克总理佩莱格里尼会晤时表示，以色列方面同意今年晚些时候与斯方在斯洛伐克举...,1,0


In [46]:
# %%
# ============================================================
# 21. Inspect false-neutral cases
# ============================================================

for prompt_col in prompt_columns_cn:

    false_neutral = results_cn[
        (results_cn[prompt_col].astype(float) == 0)
        &
        (results_cn["human_sentiment"].astype(float) != 0)
    ].copy()

    print("\n" + "=" * 100)
    print(f"FALSE NEUTRAL CASES: {prompt_col}")
    print("=" * 100)

    print(
        "Number of false-neutral cases:",
        len(false_neutral)
    )

    display(
        false_neutral[
            [
                "text",
                "human_sentiment",
                prompt_col
            ]
        ]
    )


FALSE NEUTRAL CASES: P1_CN
Number of false-neutral cases: 7


,text,human_sentiment,P1_CN
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,1,0
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,1,0
36,2022年杭州亚运会召开时，亚运区域内将全面实现自动驾驶。这是记者从13日第19届亚运会汽车...,1,0
37,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,-1,0
43,2016年8月，“费多尔”机器人原型机定型，身高180厘米，体重达160公斤，工作时功率约为...,1,0
57,在20日的传媒预览环节，展出了部分参与团队的研发项目，团队代表现场介绍其作品的理念来源、研发...,1,0
75,国务院总理李强4月7日主持召开国务院常务会议，研究推动外贸稳规模优结构的政策措施，审议通过《...,1,0



FALSE NEUTRAL CASES: P2_CN
Number of false-neutral cases: 8


,text,human_sentiment,P2_CN
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,1,0
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,1,0
23,发布的15项世界互联网领先科技成果包括：微信小程序商业模式创新、华为昇腾310芯片、蚂蚁金服...,1,0
30,美德两大汽车制造商福特和大众15日宣布多项合作协议，将联合生产皮卡和厢式货车，并探索在电动汽...,1,0
37,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,-1,0
43,2016年8月，“费多尔”机器人原型机定型，身高180厘米，体重达160公斤，工作时功率约为...,1,0
57,在20日的传媒预览环节，展出了部分参与团队的研发项目，团队代表现场介绍其作品的理念来源、研发...,1,0
75,国务院总理李强4月7日主持召开国务院常务会议，研究推动外贸稳规模优结构的政策措施，审议通过《...,1,0



FALSE NEUTRAL CASES: P3_CN
Number of false-neutral cases: 18


,text,human_sentiment,P3_CN
6,未来，芯片可被广泛应用于车辆管理、汽车导航、可穿戴设备、航海导航、ＧＩＳ数据采集、精准农业、...,1,0
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,1,0
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,1,0
21,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,1,0
25,在自动驾驶领域，大众汽车集团正在中国加速发展自动驾驶技术。目前，奥迪品牌已在北京和无锡两座城...,1,0
30,美德两大汽车制造商福特和大众15日宣布多项合作协议，将联合生产皮卡和厢式货车，并探索在电动汽...,1,0
32,产业通商资源部表示，韩国吸引外国直接投资的重点领域将包括汽车自动驾驶、医疗、智慧家庭、节能方...,1,0
33,内塔尼亚胡在与斯洛伐克总理佩莱格里尼会晤时表示，以色列方面同意今年晚些时候与斯方在斯洛伐克举...,1,0
36,2022年杭州亚运会召开时，亚运区域内将全面实现自动驾驶。这是记者从13日第19届亚运会汽车...,1,0
37,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,-1,0


In [47]:
# %%
# ============================================================
# 22. Error transition summary
# ============================================================

for prompt_col in prompt_columns_cn:

    temp = results_cn[
        results_cn[prompt_col].notna()
    ].copy()

    temp[
        "human_sentiment"
    ] = temp[
        "human_sentiment"
    ].astype(int)

    temp[
        prompt_col
    ] = temp[
        prompt_col
    ].astype(int)

    error_transition = (
        temp[
            temp["human_sentiment"]
            !=
            temp[prompt_col]
        ]
        .groupby(
            [
                "human_sentiment",
                prompt_col
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
    )

    print("\n" + "=" * 80)
    print(f"ERROR TRANSITIONS: {prompt_col}")
    print("=" * 80)

    display(error_transition)


ERROR TRANSITIONS: P1_CN


,human_sentiment,P1_CN,count
0,-1,0,1
1,0,1,12
2,1,0,6



ERROR TRANSITIONS: P2_CN


,human_sentiment,P2_CN,count
0,-1,0,1
1,0,1,11
2,1,0,7



ERROR TRANSITIONS: P3_CN


,human_sentiment,P3_CN,count
0,-1,0,1
1,0,1,8
2,1,0,17


In [48]:
# %%
# ============================================================
# 23. Save final validation workbook
# ============================================================

FINAL_ANALYSIS_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_validation_full_analysis.xlsx"
)

with pd.ExcelWriter(
    FINAL_ANALYSIS_PATH,
    engine="openpyxl"
) as writer:

    results_cn.to_excel(
        writer,
        sheet_name="article_results",
        index=False
    )

    evaluation_cn_df.to_excel(
        writer,
        sheet_name="overall_metrics",
        index=False
    )

    distribution_cn_df.to_excel(
        writer,
        sheet_name="label_counts"
    )

    distribution_pct_cn_df.to_excel(
        writer,
        sheet_name="label_percentages"
    )

print("Final analysis workbook saved:")
print(FINAL_ANALYSIS_PATH)

Final analysis workbook saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_validation_full_analysis.xlsx


In [49]:
def build_prompt_p2r_cn(text):
    return f"""
你正在为一项关于新闻媒体如何报道自动驾驶汽车的学术研究进行情感分类。

你的任务是判断下面这段新闻文本对于以下对象的整体评价方向：
自动驾驶汽车、无人驾驶汽车、自动驾驶技术，以及相关的发展、测试、部署和商业化。

请使用以下标签：

1 = Positive（正面）

当文本整体上以有利、积极或支持性的方式呈现自动驾驶汽车或自动驾驶技术时，标记为 1。

正面评价既可以是明确表达的，也可以是隐含表达的，不要求必须出现明显的正面形容词。

可能的正面信号包括：
- 明确体现技术突破、重要进展或成功研发；
- 成功完成测试、示范、部署、商业化或规模化扩展；
- 明确显示自动驾驶在安全性、效率、性能或可靠性方面取得改善；
- 明确强调其为用户、交通、社会或产业带来的实际益处；
- 政策、基础设施或制度变化明显有利于自动驾驶发展；
- 对自动驾驶未来发展表达明确乐观、支持或积极预期；
- 将某项发展描述为重要成果、领先优势、突破、成功或具有明确积极意义。

即使文本采用客观、事实性的新闻写法，只要所报道的事件本身清楚体现了自动驾驶取得成功、优势、收益或重要积极进展，也可以标记为 Positive。


0 = Neutral（中性）

当文本主要是在提供与自动驾驶有关的信息，而没有清晰的正面或负面评价方向时，标记为 0。

以下情况通常应标记为 Neutral：
- 主要报道研发、测试、合作、投资、规划、发布、参展、政策讨论或项目推进等事实，但没有明确显示这些活动取得了成功、产生了明显收益或构成重大进展；
- 只是说明某项自动驾驶技术“正在研发”“开始测试”“计划部署”“获得资格”“开展合作”或“进入某一阶段”，但没有足够证据表明结果是积极还是消极；
- 只是陈述某项技术能力、功能、项目安排或行业动态；
- 自动驾驶只是文章中的背景信息，而不是被实质性评价的对象；
- 正面和负面信息同时存在，并且没有任何一方明显占主导。

非常重要：

不要因为文本中出现“研发”“测试”“应用”“合作”“投资”“部署”“创新”“智能化”“获得资质”等词，就自动判断为 Positive。

“发生了进展”与“对自动驾驶作出正面评价”并不完全相同。

例如，仅仅报道：
- 某企业开始进行自动驾驶测试；
- 某地开放自动驾驶道路测试；
- 某公司获得测试牌照或测试资格；
- 某企业宣布研发自动驾驶技术；
- 某项目进入示范阶段；
- 某公司与另一机构开展自动驾驶合作；

如果文本没有进一步体现成功结果、明显优势、实际收益、重要突破或积极评价，应优先判断为 Neutral。


-1 = Negative（负面）

当文本整体上以不利、消极或批判性的方式呈现自动驾驶汽车或自动驾驶技术时，标记为 -1。

负面评价既可以是明确表达的，也可以是隐含表达的，不要求必须出现明显的负面形容词。

可能的负面信号包括：
- 技术故障、失效、性能不佳或可靠性问题；
- 自动驾驶相关的安全风险或安全隐患；
- 与自动驾驶系统相关的事故、碰撞、伤亡或其他有害结果；
- 测试、部署或商业化被暂停、取消、限制或明显受阻；
- 公众质疑、批评、担忧或信任下降；
- 法律纠纷、监管限制或政策障碍；
- 证据显示自动驾驶存在明显局限、不可靠性或不良后果。

即使文本采用客观、事实性的新闻写法，只要所报道的事件本身清楚体现了重大失败、安全问题、限制或发展受挫，也可以标记为 Negative。


重要判断原则：

1. 只判断文本对于自动驾驶汽车、自动驾驶技术及其发展与部署的评价方向。

2. 不要根据文章整体情绪、企业股价、公司经营状况、宏观经济环境或其他与自动驾驶无直接关系的信息进行判断。

3. 不要因为新华社新闻通常采用正式、客观、积极的报道风格，就默认将文本判断为 Positive。
必须判断文本中关于自动驾驶本身是否存在明确的积极或消极含义。

4. “技术活动本身存在”不等于“正面评价”。
研发、测试、投资、合作、规划、获得资格或开始应用，如果只是事实陈述，应标记为 Neutral。

5. 只有当文本明确体现以下至少一种情况时，才更倾向于 Positive：
- 成功结果；
- 明显技术突破；
- 已证实的性能改善；
- 实际社会、交通或产业收益；
- 明确领先优势；
- 明显有利的发展结果；
- 清楚的积极评价或乐观预期。

6. 当文本同时包含正面和负面内容时，根据整体占主导的评价方向判断。
只有在两种方向都不占明显优势时，才使用 Neutral。

7. 只根据提供的文本进行判断，不使用外部知识。

只返回一个数字标签：1、0 或 -1。

新闻文本：
{text}
"""

In [50]:
# %%
# ============================================================
# 24. Run refined P2R-CN prompt
# ============================================================

results_cn["P2R_CN"] = None

for idx, row in results_cn.iterrows():

    text = row["text"]

    print(
        f"Processing P2R-CN: "
        f"{idx + 1}/{len(results_cn)}"
    )

    results_cn.at[idx, "P2R_CN"] = classify_sentiment(
        build_prompt_p2r_cn(text)
    )

print("\nP2R-CN classification completed.")

# %%
# ============================================================
# 25. Check missing P2R-CN predictions
# ============================================================

print("Missing P2R-CN predictions:")

print(
    results_cn["P2R_CN"].isna().sum()
)

# Show rows with missing predictions if any
missing_p2r = results_cn[
    results_cn["P2R_CN"].isna()
].copy()

if len(missing_p2r) > 0:

    print("\nRows with missing predictions:")

    display(
        missing_p2r[
            [
                "text",
                "human_sentiment"
            ]
        ]
    )

else:

    print("No missing predictions.")

# %%
# ============================================================
# 26. Evaluate P2R-CN
# ============================================================

valid_p2r = results_cn[
    results_cn["P2R_CN"].notna()
].copy()

y_true_p2r = (
    valid_p2r["human_sentiment"]
    .astype(int)
)

y_pred_p2r = (
    valid_p2r["P2R_CN"]
    .astype(int)
)

p2r_metrics = {

    "prompt": "P2R_CN",

    "n": len(valid_p2r),

    "accuracy": accuracy_score(
        y_true_p2r,
        y_pred_p2r
    ),

    "macro_precision": precision_score(
        y_true_p2r,
        y_pred_p2r,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true_p2r,
        y_pred_p2r,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true_p2r,
        y_pred_p2r,
        average="macro",
        zero_division=0
    ),

    "cohen_kappa": cohen_kappa_score(
        y_true_p2r,
        y_pred_p2r
    )
}

p2r_metrics_df = pd.DataFrame(
    [p2r_metrics]
)

print("\nP2R-CN overall performance:")

display(
    p2r_metrics_df.round(4)
)

# %%
# ============================================================
# 27. Compare P1 / P2 / P3 / P2R
# ============================================================

prompt_columns_cn_updated = [
    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN"
]

evaluation_rows_cn_updated = []

for prompt_col in prompt_columns_cn_updated:

    valid = results_cn[
        results_cn[prompt_col].notna()
    ].copy()

    y_true = (
        valid["human_sentiment"]
        .astype(int)
    )

    y_pred = (
        valid[prompt_col]
        .astype(int)
    )

    evaluation_rows_cn_updated.append({

        "prompt": prompt_col,

        "n": len(valid),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "cohen_kappa": cohen_kappa_score(
            y_true,
            y_pred
        )
    })


evaluation_cn_updated_df = pd.DataFrame(
    evaluation_rows_cn_updated
)

print("\nOverall prompt comparison:")

display(
    evaluation_cn_updated_df.round(4)
)

# %%
# ============================================================
# 28. Class-level performance for P2R-CN
# ============================================================

print("\n" + "=" * 100)
print("CLASS-LEVEL PERFORMANCE: P2R_CN")
print("=" * 100)

p2r_report = classification_report(
    y_true_p2r,
    y_pred_p2r,
    labels=[-1, 0, 1],
    target_names=[
        "Negative",
        "Neutral",
        "Positive"
    ],
    output_dict=True,
    zero_division=0
)

p2r_report_df = pd.DataFrame(
    p2r_report
).T

display(
    p2r_report_df[
        [
            "precision",
            "recall",
            "f1-score",
            "support"
        ]
    ].round(4)
)

# %%
# ============================================================
# 29. P2R-CN confusion matrix
# ============================================================

p2r_cm = confusion_matrix(
    y_true_p2r,
    y_pred_p2r,
    labels=[-1, 0, 1]
)

p2r_cm_df = pd.DataFrame(
    p2r_cm,
    index=[
        "Human Negative",
        "Human Neutral",
        "Human Positive"
    ],
    columns=[
        "Pred Negative",
        "Pred Neutral",
        "Pred Positive"
    ]
)

print("\nP2R-CN confusion matrix:")

display(
    p2r_cm_df
)

# %%
# ============================================================
# 30. Compare label distributions
# ============================================================

columns_to_check_cn_updated = [
    "human_sentiment",
    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN"
]

distribution_cn_updated = {}

for col in columns_to_check_cn_updated:

    counts = (
        results_cn[col]
        .dropna()
        .astype(int)
        .value_counts()
        .reindex(
            [-1, 0, 1],
            fill_value=0
        )
    )

    distribution_cn_updated[col] = counts


distribution_cn_updated_df = pd.DataFrame(
    distribution_cn_updated
)

distribution_cn_updated_df.index = [
    "Negative (-1)",
    "Neutral (0)",
    "Positive (1)"
]

print("\nLabel counts:")

display(
    distribution_cn_updated_df
)

# %%
# ============================================================
# 31. Compare label percentages
# ============================================================

distribution_pct_cn_updated_df = (
    distribution_cn_updated_df
    /
    distribution_cn_updated_df.sum(axis=0)
    *
    100
)

print("\nLabel percentages:")

display(
    distribution_pct_cn_updated_df.round(2)
)

# %%
# ============================================================
# 32. P2R-CN error analysis
# ============================================================

p2r_errors = results_cn[
    results_cn["P2R_CN"].notna()
].copy()

p2r_errors = p2r_errors[
    p2r_errors["P2R_CN"].astype(int)
    !=
    p2r_errors["human_sentiment"].astype(int)
].copy()

print("\n" + "=" * 100)
print("ERRORS FOR P2R_CN")
print("=" * 100)

print(
    "Number of errors:",
    len(p2r_errors)
)

display(
    p2r_errors[
        [
            "text",
            "human_sentiment",
            "P2R_CN"
        ]
    ]
)

# %%
# ============================================================
# 33. P2R-CN error transitions
# ============================================================

p2r_transition = (
    p2r_errors
    .assign(
        human_sentiment=lambda x:
            x["human_sentiment"].astype(int),

        P2R_CN=lambda x:
            x["P2R_CN"].astype(int)
    )
    .groupby(
        [
            "human_sentiment",
            "P2R_CN"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

print("\nP2R-CN error transitions:")

display(
    p2r_transition
)

# %%
# ============================================================
# 34. Specifically inspect Neutral -> Positive errors
# ============================================================

neutral_to_positive_p2r = results_cn[
    results_cn["P2R_CN"].notna()
].copy()

neutral_to_positive_p2r = neutral_to_positive_p2r[
    (
        neutral_to_positive_p2r[
            "human_sentiment"
        ].astype(int) == 0
    )
    &
    (
        neutral_to_positive_p2r[
            "P2R_CN"
        ].astype(int) == 1
    )
].copy()

print("\n" + "=" * 100)
print("P2R-CN: HUMAN NEUTRAL -> MODEL POSITIVE")
print("=" * 100)

print(
    "Number of Neutral -> Positive errors:",
    len(neutral_to_positive_p2r)
)

display(
    neutral_to_positive_p2r[
        [
            "text",
            "human_sentiment",
            "P2R_CN"
        ]
    ]
)

# %%
# ============================================================
# 35. Specifically inspect Positive -> Neutral errors
# ============================================================

positive_to_neutral_p2r = results_cn[
    results_cn["P2R_CN"].notna()
].copy()

positive_to_neutral_p2r = positive_to_neutral_p2r[
    (
        positive_to_neutral_p2r[
            "human_sentiment"
        ].astype(int) == 1
    )
    &
    (
        positive_to_neutral_p2r[
            "P2R_CN"
        ].astype(int) == 0
    )
].copy()

print("\n" + "=" * 100)
print("P2R-CN: HUMAN POSITIVE -> MODEL NEUTRAL")
print("=" * 100)

print(
    "Number of Positive -> Neutral errors:",
    len(positive_to_neutral_p2r)
)

display(
    positive_to_neutral_p2r[
        [
            "text",
            "human_sentiment",
            "P2R_CN"
        ]
    ]
)

# %%
# ============================================================
# 36. Direct comparison between P2-CN and P2R-CN
# ============================================================

comparison_p2_p2r = results_cn[
    results_cn["P2_CN"].notna()
    &
    results_cn["P2R_CN"].notna()
].copy()

comparison_p2_p2r["P2_correct"] = (
    comparison_p2_p2r["P2_CN"].astype(int)
    ==
    comparison_p2_p2r["human_sentiment"].astype(int)
)

comparison_p2_p2r["P2R_correct"] = (
    comparison_p2_p2r["P2R_CN"].astype(int)
    ==
    comparison_p2_p2r["human_sentiment"].astype(int)
)

# Cases fixed by P2R
fixed_by_p2r = comparison_p2_p2r[
    (~comparison_p2_p2r["P2_correct"])
    &
    (comparison_p2_p2r["P2R_correct"])
].copy()

# Cases made worse by P2R
worsened_by_p2r = comparison_p2_p2r[
    (comparison_p2_p2r["P2_correct"])
    &
    (~comparison_p2_p2r["P2R_correct"])
].copy()


print("\n" + "=" * 100)
print("CASES FIXED BY P2R-CN")
print("=" * 100)

print(
    "Number fixed:",
    len(fixed_by_p2r)
)

display(
    fixed_by_p2r[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_CN"
        ]
    ]
)


print("\n" + "=" * 100)
print("CASES MADE WORSE BY P2R-CN")
print("=" * 100)

print(
    "Number worsened:",
    len(worsened_by_p2r)
)

display(
    worsened_by_p2r[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_CN"
        ]
    ]
)

# %%
# ============================================================
# 37. Summary: P2 vs P2R
# ============================================================

p2_vs_p2r_summary = pd.DataFrame({

    "metric": [
        "Cases fixed by P2R",
        "Cases worsened by P2R",
        "Net improvement"
    ],

    "value": [
        len(fixed_by_p2r),
        len(worsened_by_p2r),
        len(fixed_by_p2r)
        -
        len(worsened_by_p2r)
    ]
})

display(
    p2_vs_p2r_summary
)

# %%
# ============================================================
# 38. Save updated evaluation metrics
# ============================================================

UPDATED_EVALUATION_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_evaluation_metrics_with_P2R.xlsx"
)

evaluation_cn_updated_df.to_excel(
    UPDATED_EVALUATION_PATH,
    index=False
)

print("Updated evaluation metrics saved:")
print(UPDATED_EVALUATION_PATH)

# %%
# ============================================================
# 39. Save final P2R validation workbook
# ============================================================

FINAL_P2R_ANALYSIS_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_validation_with_P2R_full_analysis.xlsx"
)

with pd.ExcelWriter(
    FINAL_P2R_ANALYSIS_PATH,
    engine="openpyxl"
) as writer:

    # Article-level predictions
    results_cn.to_excel(
        writer,
        sheet_name="article_results",
        index=False
    )

    # Overall metrics
    evaluation_cn_updated_df.to_excel(
        writer,
        sheet_name="overall_metrics",
        index=False
    )

    # P2R class-level metrics
    p2r_report_df.to_excel(
        writer,
        sheet_name="P2R_class_metrics"
    )

    # Label counts
    distribution_cn_updated_df.to_excel(
        writer,
        sheet_name="label_counts"
    )

    # Label percentages
    distribution_pct_cn_updated_df.to_excel(
        writer,
        sheet_name="label_percentages"
    )

    # Confusion matrix
    p2r_cm_df.to_excel(
        writer,
        sheet_name="P2R_confusion_matrix"
    )

    # All P2R errors
    p2r_errors.to_excel(
        writer,
        sheet_name="P2R_errors",
        index=False
    )

    # Error transitions
    p2r_transition.to_excel(
        writer,
        sheet_name="P2R_error_transitions",
        index=False
    )

    # Neutral -> Positive errors
    neutral_to_positive_p2r.to_excel(
        writer,
        sheet_name="neutral_to_positive",
        index=False
    )

    # Positive -> Neutral errors
    positive_to_neutral_p2r.to_excel(
        writer,
        sheet_name="positive_to_neutral",
        index=False
    )

    # Cases fixed relative to P2
    fixed_by_p2r.to_excel(
        writer,
        sheet_name="fixed_vs_P2",
        index=False
    )

    # Cases worsened relative to P2
    worsened_by_p2r.to_excel(
        writer,
        sheet_name="worsened_vs_P2",
        index=False
    )

    # Simple summary
    p2_vs_p2r_summary.to_excel(
        writer,
        sheet_name="P2_vs_P2R_summary",
        index=False
    )


print("\nFinal P2R analysis workbook saved:")
print(FINAL_P2R_ANALYSIS_PATH)

Processing P2R-CN: 1/116
Processing P2R-CN: 2/116
Processing P2R-CN: 3/116
Processing P2R-CN: 4/116
Processing P2R-CN: 5/116
Processing P2R-CN: 6/116
Processing P2R-CN: 7/116
Processing P2R-CN: 8/116
Processing P2R-CN: 9/116
Processing P2R-CN: 10/116
Processing P2R-CN: 11/116
Processing P2R-CN: 12/116
Processing P2R-CN: 13/116
Processing P2R-CN: 14/116
Processing P2R-CN: 15/116
Processing P2R-CN: 16/116
Processing P2R-CN: 17/116
Processing P2R-CN: 18/116
Processing P2R-CN: 19/116
Processing P2R-CN: 20/116
Processing P2R-CN: 21/116
Processing P2R-CN: 22/116
Processing P2R-CN: 23/116
Processing P2R-CN: 24/116
Processing P2R-CN: 25/116
Processing P2R-CN: 26/116
Processing P2R-CN: 27/116
Processing P2R-CN: 28/116
Processing P2R-CN: 29/116
Processing P2R-CN: 30/116
Processing P2R-CN: 31/116
Processing P2R-CN: 32/116
Processing P2R-CN: 33/116
Processing P2R-CN: 34/116
Processing P2R-CN: 35/116
Processing P2R-CN: 36/116
Processing P2R-CN: 37/116
Processing P2R-CN: 38/116
Processing P2R-CN: 39

,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P2R_CN,116,0.7759,0.7857,0.6551,0.686,0.4237



Overall prompt comparison:


,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P1_CN,116,0.8362,0.8291,0.6374,0.7025,0.4769
1,P2_CN,116,0.8362,0.8281,0.6483,0.7093,0.4932
2,P3_CN,116,0.7759,0.7857,0.6551,0.6860,0.4237
3,P2R_CN,116,0.7759,0.7857,0.6551,0.6860,0.4237



CLASS-LEVEL PERFORMANCE: P2R_CN


...

,text,human_sentiment,P2_CN,P2R_CN
10,英国目前正借助工业界和学术界的力量推动多项创新技术在国防领域的应用，此前已和美国合作，实地测...,0,1,0
23,发布的15项世界互联网领先科技成果包括：微信小程序商业模式创新、华为昇腾310芯片、蚂蚁金服...,1,0,1
43,2016年8月，“费多尔”机器人原型机定型，身高180厘米，体重达160公斤，工作时功率约为...,1,0,1
71,据《日本经济新闻》中文版“日经中文网”21日报道，横滨国立大学正在研发可在海上追随台风进行发...,0,1,0
85,在重庆，李强走进永川区城东片区综合改造提升项目现场，听取重庆推动成渝地区双城经济圈建设等情况...,0,1,0
110,这名负责人介绍，规范低空管理系统建设，要坚持严控风险，一体打造低空空管、联合监管等核心功能，...,0,1,0



CASES MADE WORSE BY P2R-CN
Number worsened: 13


,text,human_sentiment,P2_CN,P2R_CN
1,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,1,1,0
2,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,0,0,1
6,未来，芯片可被广泛应用于车辆管理、汽车导航、可穿戴设备、航海导航、ＧＩＳ数据采集、精准农业、...,1,1,0
21,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,1,1,0
25,在自动驾驶领域，大众汽车集团正在中国加速发展自动驾驶技术。目前，奥迪品牌已在北京和无锡两座城...,1,1,0
40,王炳南说，与首届进博会相比，第二届进博会新设消费品新品专区，增加养老等题材，新增室外汽车“无...,1,1,0
41,滴滴出行与腾讯建立的互联网安全联合实验室将专注三个领域：一是信息安全领域的联合能力建设；二是...,1,1,0
46,他表示，宝马非常看好中国市场，与长城汽车设立合资公司生产MINI品牌电动车。宝马还与百度、阿...,1,1,0
78,在宝马世界展示中心，李强在宝马公司董事长齐普策陪同下参观了智能网联汽车、新能源汽车等展区，详...,1,1,0
88,这是这家德国老牌车企在华的又一次增资布局。进入中国40年来，大众在中国建设了39家工厂，拥有...,1,1,0


,metric,value
0,Cases fixed by P2R,6
1,Cases worsened by P2R,13
2,Net improvement,-7


Updated evaluation metrics saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_evaluation_metrics_with_P2R.xlsx

Final P2R analysis workbook saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_validation_with_P2R_full_analysis.xlsx


In [51]:
# %%
# ============================================================
# 40. P2R2-CN: Lightly Refined P2 Prompt
# ============================================================

def build_prompt_p2r2_cn(text):
    return f"""
你正在为一项关于自动驾驶新闻报道的学术研究进行情感分类。

你的任务是判断以下新闻文本对自动驾驶汽车、无人驾驶汽车、
自动驾驶技术及其发展、测试、部署和商业化所表达的总体评价倾向。

请使用以下标签：

1 = Positive（正面）

当文本整体上以积极、有利或支持性的方式呈现自动驾驶汽车
或自动驾驶技术时，标记为 1。

正面评价可以是明确表达的，也可以通过所报道的发展本身隐含体现。
不要求必须出现明显的正面形容词。

可能的正面信号包括：

- 技术进步、技术突破或研发成果；
- 成功测试、成功部署、示范应用、商业化或规模扩张；
- 自动驾驶技术能力、安全性、效率、性能或可靠性的改善；
- 对用户、交通、社会或产业带来的明显益处；
- 有利于自动驾驶发展的政策、基础设施或制度支持；
- 企业、城市或行业在自动驾驶发展方面取得实质性推进；
- 明确积极、支持或乐观的未来预期。

重要：

客观、事实性的新闻报道也可以是 Positive。

如果文本所报道的事实本身明显代表自动驾驶取得了技术进展、
发展推进、成功成果、实际应用或有利发展，即使没有明确使用
“成功”“突破”“优秀”等评价词，也可以判断为 Positive。


0 = Neutral（中性）

当文本主要是在提供与自动驾驶有关的信息，
但没有形成清晰的正面或负面评价方向时，标记为 0。

Neutral 通常包括：

- 主要提供背景信息、基本事实或技术说明；
- 仅仅宣布研究计划、合作计划、未来规划或意向，
  但尚未体现实际进展或结果；
- 仅描述某项测试、项目、会议、政策讨论或企业行为的存在，
  但无法判断其对自动驾驶发展产生明显有利或不利影响；
- 自动驾驶只是文章中的背景信息；
- 正面和负面因素同时存在，并且没有一方明显占主导。

需要注意：

不要仅仅因为出现“研发”“测试”“合作”“投资”“部署”
或“获得资质”等词，就自动判断为 Positive。

但是，也不要因为这些内容采用事实性新闻语言，
就自动判断为 Neutral。

应判断这些事实在具体上下文中是否实际体现了：

- 自动驾驶技术能力的提升；
- 研发或测试取得实质性进展；
- 部署范围或商业应用扩大；
- 发展条件明显改善；
- 或其他明显有利于自动驾驶发展的结果。

如果存在这些含义，应倾向于判断为 Positive。

如果只是宣布、计划、讨论、尝试或一般性事实，
而没有体现明确的发展结果，则应倾向于 Neutral。


-1 = Negative（负面）

当文本整体上以消极、不利或批判性的方式呈现自动驾驶汽车
或自动驾驶技术时，标记为 -1。

负面评价可以是明确表达的，也可以通过所报道的发展本身隐含体现。

可能的负面信号包括：

- 技术失败、功能失效或性能不佳；
- 安全风险、安全隐患或可靠性问题；
- 与自动驾驶系统表现相关的事故、碰撞、伤亡或其他有害结果；
- 测试、部署或商业化受到暂停、限制、取消或明显阻碍；
- 公众质疑、批评、担忧或信任下降；
- 法律、监管或政策障碍；
- 明显的发展受挫或不利后果。

即使文本采用客观、事实性的新闻语言，
如果所报道的事件本身明显代表自动驾驶失败、受阻、
受到限制或产生不利后果，也可以判断为 Negative。


请按照以下原则判断：

1. 只判断文本对自动驾驶汽车、自动驾驶技术及其发展与部署的评价倾向。

2. 不要根据文章的一般情绪色彩、
企业股价、财务表现、公司整体经营状况
或其他与自动驾驶无直接关系的信息判断。

3. 客观新闻语言不等于 Neutral。
应根据文本所报道的发展及其含义判断评价方向。

4. “出现了某项活动”与“取得了积极进展”需要区分。

例如：
仅宣布未来研发计划、合作意向或测试安排，
通常可以判断为 Neutral。

但如果文本表明测试已经成功完成、
技术能力得到提升、实际应用扩大、
取得重要发展成果或形成明显有利条件，
则可以判断为 Positive。

5. 不要求 Positive 必须包含明确主观评价词。
事实本身可以形成积极或消极的评价方向。

6. 当文本同时包含正面和负面内容时，
根据整体占主导的评价方向判断。
只有当两种方向均不明显占主导时，才判断为 Neutral。

7. 只能依据所提供的文本进行判断。
不要使用外部知识。

只返回一个数字标签：1、0 或 -1。

新闻文本：
{text}
""".strip()

# %%
# ============================================================
# 41. Run P2R2-CN
# ============================================================

results_cn["P2R2_CN"] = None

for idx, row in results_cn.iterrows():

    text = row["text"]

    print(
        f"Processing P2R2-CN: "
        f"{idx + 1}/{len(results_cn)}"
    )

    results_cn.at[idx, "P2R2_CN"] = classify_sentiment(
        build_prompt_p2r2_cn(text)
    )

print("\nP2R2-CN classification completed.")

# %%
# ============================================================
# 42. Check missing P2R2-CN predictions
# ============================================================

print("Missing P2R2-CN predictions:")

print(
    results_cn["P2R2_CN"].isna().sum()
)

missing_p2r2 = results_cn[
    results_cn["P2R2_CN"].isna()
].copy()

if len(missing_p2r2) > 0:

    display(
        missing_p2r2[
            [
                "text",
                "human_sentiment"
            ]
        ]
    )

else:

    print("No missing predictions.")

# %%
# ============================================================
# 43. Evaluate P2R2-CN
# ============================================================

valid_p2r2 = results_cn[
    results_cn["P2R2_CN"].notna()
].copy()

y_true_p2r2 = (
    valid_p2r2["human_sentiment"]
    .astype(int)
)

y_pred_p2r2 = (
    valid_p2r2["P2R2_CN"]
    .astype(int)
)

p2r2_metrics = {

    "prompt": "P2R2_CN",

    "n": len(valid_p2r2),

    "accuracy": accuracy_score(
        y_true_p2r2,
        y_pred_p2r2
    ),

    "macro_precision": precision_score(
        y_true_p2r2,
        y_pred_p2r2,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true_p2r2,
        y_pred_p2r2,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true_p2r2,
        y_pred_p2r2,
        average="macro",
        zero_division=0
    ),

    "cohen_kappa": cohen_kappa_score(
        y_true_p2r2,
        y_pred_p2r2
    )
}

p2r2_metrics_df = pd.DataFrame(
    [p2r2_metrics]
)

print("\nP2R2-CN overall performance:")

display(
    p2r2_metrics_df.round(4)
)

# %%
# ============================================================
# 44. Compare all prompts
# ============================================================

prompt_columns_cn_final = [
    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN",
    "P2R2_CN"
]

evaluation_rows_cn_final = []

for prompt_col in prompt_columns_cn_final:

    valid = results_cn[
        results_cn[prompt_col].notna()
    ].copy()

    y_true = (
        valid["human_sentiment"]
        .astype(int)
    )

    y_pred = (
        valid[prompt_col]
        .astype(int)
    )

    evaluation_rows_cn_final.append({

        "prompt": prompt_col,

        "n": len(valid),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "cohen_kappa": cohen_kappa_score(
            y_true,
            y_pred
        )
    })


evaluation_cn_final_df = pd.DataFrame(
    evaluation_rows_cn_final
)

print("\nFinal overall prompt comparison:")

display(
    evaluation_cn_final_df.round(4)
)

# %%
# ============================================================
# 45. Class-level performance for P2R2-CN
# ============================================================

print("\n" + "=" * 100)
print("CLASS-LEVEL PERFORMANCE: P2R2_CN")
print("=" * 100)

p2r2_report = classification_report(
    y_true_p2r2,
    y_pred_p2r2,
    labels=[-1, 0, 1],
    target_names=[
        "Negative",
        "Neutral",
        "Positive"
    ],
    output_dict=True,
    zero_division=0
)

p2r2_report_df = pd.DataFrame(
    p2r2_report
).T

display(
    p2r2_report_df[
        [
            "precision",
            "recall",
            "f1-score",
            "support"
        ]
    ].round(4)
)

# %%
# ============================================================
# 46. P2R2-CN confusion matrix
# ============================================================

p2r2_cm = confusion_matrix(
    y_true_p2r2,
    y_pred_p2r2,
    labels=[-1, 0, 1]
)

p2r2_cm_df = pd.DataFrame(
    p2r2_cm,
    index=[
        "Human Negative",
        "Human Neutral",
        "Human Positive"
    ],
    columns=[
        "Pred Negative",
        "Pred Neutral",
        "Pred Positive"
    ]
)

print("\nP2R2-CN confusion matrix:")

display(
    p2r2_cm_df
)

# %%
# ============================================================
# 47. Compare label distributions
# ============================================================

columns_to_check_cn_final = [
    "human_sentiment",
    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN",
    "P2R2_CN"
]

distribution_cn_final = {}

for col in columns_to_check_cn_final:

    counts = (
        results_cn[col]
        .dropna()
        .astype(int)
        .value_counts()
        .reindex(
            [-1, 0, 1],
            fill_value=0
        )
    )

    distribution_cn_final[col] = counts


distribution_cn_final_df = pd.DataFrame(
    distribution_cn_final
)

distribution_cn_final_df.index = [
    "Negative (-1)",
    "Neutral (0)",
    "Positive (1)"
]

print("\nLabel counts:")

display(
    distribution_cn_final_df
)

# %%
# ============================================================
# 48. Compare label percentages
# ============================================================

distribution_pct_cn_final_df = (
    distribution_cn_final_df
    /
    distribution_cn_final_df.sum(axis=0)
    *
    100
)

print("\nLabel percentages:")

display(
    distribution_pct_cn_final_df.round(2)
)

# %%
# ============================================================
# 49. P2R2-CN error analysis
# ============================================================

p2r2_errors = results_cn[
    results_cn["P2R2_CN"].notna()
].copy()

p2r2_errors = p2r2_errors[
    p2r2_errors["P2R2_CN"].astype(int)
    !=
    p2r2_errors["human_sentiment"].astype(int)
].copy()

print("\n" + "=" * 100)
print("ERRORS FOR P2R2_CN")
print("=" * 100)

print(
    "Number of errors:",
    len(p2r2_errors)
)

display(
    p2r2_errors[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_CN",
            "P2R2_CN"
        ]
    ]
)

# %%
# ============================================================
# 50. P2R2-CN error transitions
# ============================================================

p2r2_transition = (
    p2r2_errors
    .assign(
        human_sentiment=lambda x:
            x["human_sentiment"].astype(int),

        P2R2_CN=lambda x:
            x["P2R2_CN"].astype(int)
    )
    .groupby(
        [
            "human_sentiment",
            "P2R2_CN"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

print("\nP2R2-CN error transitions:")

display(
    p2r2_transition
)

# %%
# ============================================================
# 51. Inspect Neutral -> Positive errors
# ============================================================

neutral_to_positive_p2r2 = results_cn[
    results_cn["P2R2_CN"].notna()
].copy()

neutral_to_positive_p2r2 = neutral_to_positive_p2r2[
    (
        neutral_to_positive_p2r2[
            "human_sentiment"
        ].astype(int) == 0
    )
    &
    (
        neutral_to_positive_p2r2[
            "P2R2_CN"
        ].astype(int) == 1
    )
].copy()

print("\n" + "=" * 100)
print("P2R2-CN: HUMAN NEUTRAL -> MODEL POSITIVE")
print("=" * 100)

print(
    "Number of Neutral -> Positive errors:",
    len(neutral_to_positive_p2r2)
)

display(
    neutral_to_positive_p2r2[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R2_CN"
        ]
    ]
)

# %%
# ============================================================
# 52. Inspect Positive -> Neutral errors
# ============================================================

positive_to_neutral_p2r2 = results_cn[
    results_cn["P2R2_CN"].notna()
].copy()

positive_to_neutral_p2r2 = positive_to_neutral_p2r2[
    (
        positive_to_neutral_p2r2[
            "human_sentiment"
        ].astype(int) == 1
    )
    &
    (
        positive_to_neutral_p2r2[
            "P2R2_CN"
        ].astype(int) == 0
    )
].copy()

print("\n" + "=" * 100)
print("P2R2-CN: HUMAN POSITIVE -> MODEL NEUTRAL")
print("=" * 100)

print(
    "Number of Positive -> Neutral errors:",
    len(positive_to_neutral_p2r2)
)

display(
    positive_to_neutral_p2r2[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R2_CN"
        ]
    ]
)

# %%
# ============================================================
# 53. Direct comparison: P2 vs P2R2
# ============================================================

comparison_p2_p2r2 = results_cn[
    results_cn["P2_CN"].notna()
    &
    results_cn["P2R2_CN"].notna()
].copy()

comparison_p2_p2r2["P2_correct"] = (
    comparison_p2_p2r2["P2_CN"].astype(int)
    ==
    comparison_p2_p2r2["human_sentiment"].astype(int)
)

comparison_p2_p2r2["P2R2_correct"] = (
    comparison_p2_p2r2["P2R2_CN"].astype(int)
    ==
    comparison_p2_p2r2["human_sentiment"].astype(int)
)

fixed_by_p2r2 = comparison_p2_p2r2[
    (~comparison_p2_p2r2["P2_correct"])
    &
    (comparison_p2_p2r2["P2R2_correct"])
].copy()

worsened_by_p2r2 = comparison_p2_p2r2[
    (comparison_p2_p2r2["P2_correct"])
    &
    (~comparison_p2_p2r2["P2R2_correct"])
].copy()

print("\n" + "=" * 100)
print("CASES FIXED BY P2R2-CN")
print("=" * 100)

print(
    "Number fixed:",
    len(fixed_by_p2r2)
)

display(
    fixed_by_p2r2[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R2_CN"
        ]
    ]
)


print("\n" + "=" * 100)
print("CASES MADE WORSE BY P2R2-CN")
print("=" * 100)

print(
    "Number worsened:",
    len(worsened_by_p2r2)
)

display(
    worsened_by_p2r2[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R2_CN"
        ]
    ]
)

# %%
# ============================================================
# 54. Summary: P2 vs P2R2
# ============================================================

p2_vs_p2r2_summary = pd.DataFrame({

    "metric": [
        "Cases fixed by P2R2",
        "Cases worsened by P2R2",
        "Net improvement"
    ],

    "value": [
        len(fixed_by_p2r2),
        len(worsened_by_p2r2),
        len(fixed_by_p2r2)
        -
        len(worsened_by_p2r2)
    ]
})

print("\nP2 vs P2R2 summary:")

display(
    p2_vs_p2r2_summary
)

# %%
# ============================================================
# 55. Save final workbook with P2R2
# ============================================================

FINAL_P2R2_ANALYSIS_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_validation_with_P2R2_full_analysis.xlsx"
)

with pd.ExcelWriter(
    FINAL_P2R2_ANALYSIS_PATH,
    engine="openpyxl"
) as writer:

    results_cn.to_excel(
        writer,
        sheet_name="article_results",
        index=False
    )

    evaluation_cn_final_df.to_excel(
        writer,
        sheet_name="overall_metrics",
        index=False
    )

    p2r2_report_df.to_excel(
        writer,
        sheet_name="P2R2_class_metrics"
    )

    distribution_cn_final_df.to_excel(
        writer,
        sheet_name="label_counts"
    )

    distribution_pct_cn_final_df.to_excel(
        writer,
        sheet_name="label_percentages"
    )

    p2r2_cm_df.to_excel(
        writer,
        sheet_name="P2R2_confusion_matrix"
    )

    p2r2_errors.to_excel(
        writer,
        sheet_name="P2R2_errors",
        index=False
    )

    p2r2_transition.to_excel(
        writer,
        sheet_name="P2R2_error_transitions",
        index=False
    )

    neutral_to_positive_p2r2.to_excel(
        writer,
        sheet_name="neutral_to_positive",
        index=False
    )

    positive_to_neutral_p2r2.to_excel(
        writer,
        sheet_name="positive_to_neutral",
        index=False
    )

    fixed_by_p2r2.to_excel(
        writer,
        sheet_name="fixed_vs_P2",
        index=False
    )

    worsened_by_p2r2.to_excel(
        writer,
        sheet_name="worsened_vs_P2",
        index=False
    )

    p2_vs_p2r2_summary.to_excel(
        writer,
        sheet_name="P2_vs_P2R2_summary",
        index=False
    )

print("\nFinal P2R2 analysis workbook saved:")
print(FINAL_P2R2_ANALYSIS_PATH)

Processing P2R2-CN: 1/116
Processing P2R2-CN: 2/116
Processing P2R2-CN: 3/116
Processing P2R2-CN: 4/116
Processing P2R2-CN: 5/116
Processing P2R2-CN: 6/116
Processing P2R2-CN: 7/116
Processing P2R2-CN: 8/116
Processing P2R2-CN: 9/116
Processing P2R2-CN: 10/116
Processing P2R2-CN: 11/116
Processing P2R2-CN: 12/116
Processing P2R2-CN: 13/116
Processing P2R2-CN: 14/116
Processing P2R2-CN: 15/116
Processing P2R2-CN: 16/116
Processing P2R2-CN: 17/116
Processing P2R2-CN: 18/116
Processing P2R2-CN: 19/116
Processing P2R2-CN: 20/116
Processing P2R2-CN: 21/116
Processing P2R2-CN: 22/116
Processing P2R2-CN: 23/116
Processing P2R2-CN: 24/116
Processing P2R2-CN: 25/116
Processing P2R2-CN: 26/116
Processing P2R2-CN: 27/116
Processing P2R2-CN: 28/116
Processing P2R2-CN: 29/116
Processing P2R2-CN: 30/116
Processing P2R2-CN: 31/116
Processing P2R2-CN: 32/116
Processing P2R2-CN: 33/116
Processing P2R2-CN: 34/116
Processing P2R2-CN: 35/116
Processing P2R2-CN: 36/116
Processing P2R2-CN: 37/116
Processing

,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P2R2_CN,116,0.8362,0.8403,0.605,0.677,0.4208



Final overall prompt comparison:


,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P1_CN,116,0.8362,0.8291,0.6374,0.7025,0.4769
1,P2_CN,116,0.8362,0.8281,0.6483,0.7093,0.4932
2,P3_CN,116,0.7759,0.7857,0.6551,0.6860,0.4237
3,P2R_CN,116,0.7759,0.7857,0.6551,0.6860,0.4237
4,P2R2_CN,116,0.8362,0.8403,0.6050,0.6770,0.4208



CLASS-LEVEL PERFORMANCE: P2R2_CN


...


CASES MADE WORSE BY P2R2-CN
Number worsened: 7


,text,human_sentiment,P2_CN,P2R2_CN
2,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,0,0,1
6,未来，芯片可被广泛应用于车辆管理、汽车导航、可穿戴设备、航海导航、ＧＩＳ数据采集、精准农业、...,1,1,0
15,谷歌母公司“字母表”旗下的“出行新方式”（Ｗａｙｍｏ）公司９日宣布，将从下周开始在美国亚特兰...,0,0,1
21,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,1,1,0
27,在北美轿车市场萎缩的情况下，通用汽车不久前宣布在美国和加拿大大幅减员，同时削减滞销车型的生产...,0,0,1
35,中国重要的国产车企——重庆长安汽车股份有限公司，也通过中欧班列把德国的汽车零部件运回来，用于...,0,0,1
54,3D扫描定制西服、无人驾驶汽车驶上公路、智能生产车间……在中国重庆举行的中国国际智能产业博览...,0,0,1



P2 vs P2R2 summary:


,metric,value
0,Cases fixed by P2R2,7
1,Cases worsened by P2R2,7
2,Net improvement,0



Final P2R2 analysis workbook saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_validation_with_P2R2_full_analysis.xlsx


In [52]:
# %%
# ============================================================
# 56. Matched P2R-CN prompt
#     Exact conceptual match to English P2R
# ============================================================

def build_prompt_p2r_matched_cn(text):
    return f"""
你正在为一项关于自动驾驶汽车新闻报道的学术研究进行情感分类。

你的任务是判断以下新闻文本对于自动驾驶汽车、自动驾驶技术，
以及相关发展和部署的整体评价方向。

请使用以下标签：

1 = Positive（正面）

当文本整体上以有利的方式呈现自动驾驶汽车或自动驾驶技术时，
判断为正面。

正面情感可以通过明确表达体现，也可以通过隐含方式体现。
不要求文本必须包含明确的正面词语。

可能的正面信号包括：
- 技术进步或成功发展；
- 成功测试、部署、商业化或扩张；
- 已经体现出的安全性或性能改善；
- 对用户、社会、交通出行或产业带来的益处；
- 有利于自动驾驶发展的支持性变化；
- 对未来发展的乐观预期。

因此，即使文本只是以事实性方式描述某项成功进展或有利发展，
只要其整体含义明显有利于自动驾驶，
仍然可以判断为 Positive。


0 = Neutral（中性）

当文本主要是在提供与自动驾驶有关的信息，
但没有可以辨别出的明确正面或负面评价方向时，
判断为中性。

以下情况可以判断为 Neutral：
- 文本主要报道事实，但并未体现明确的进展、益处、风险、失败或发展受挫；
- 正面和负面因素同时存在，并且没有任何一方明显占主导；
- 自动驾驶只是作为背景信息被提及，而没有被实质性评价。

不要仅仅因为文本采用客观、事实性的新闻写作方式，
就将其判断为 Neutral。

新闻报道仍然可以通过它所强调的发展、结果和后果，
传达正面或负面的评价方向。


-1 = Negative（负面）

当文本整体上以不利的方式呈现自动驾驶汽车或自动驾驶技术时，
判断为负面。

负面情感可以通过明确表达体现，也可以通过隐含方式体现。
不要求文本必须包含明确的负面词语。

可能的负面信号包括：
- 技术失败或性能表现不佳；
- 与自动驾驶相关的安全问题或风险；
- 与自动驾驶系统表现相关的事故或有害后果；
- 测试或部署被暂停、限制、取消，或出现发展受挫；
- 批评、公众担忧、法律问题或监管障碍；
- 表明技术存在局限、不可靠性或不利后果的证据。

因此，即使文本只是以事实性方式描述某项重大失败、
安全问题、限制或发展受挫，
只要其整体含义明显不利于自动驾驶，
仍然可以判断为 Negative。


重要：

只判断文本对于自动驾驶汽车、自动驾驶技术，
以及相关发展和部署的评价方向。

不要根据新闻故事整体的一般情绪色彩、
公司的股价表现，
或者与自动驾驶无关的商业发展进行判断。

当文本同时包含正面和负面因素时，
根据占主导地位的整体评价方向进行分类。

只有当正面和负面方向都没有明显占主导时，
才使用 Neutral。

只能根据所提供的文本作出判断。
不要使用外部知识。

只返回一个数字标签：1、0 或 -1。

新闻文本：
{text}
""".strip()


# %%
# ============================================================
# 57. Run P2R_MATCHED_CN
# ============================================================

results_cn["P2R_MATCHED_CN"] = None

for idx, row in results_cn.iterrows():

    text = row["text"]

    print(
        f"Processing P2R_MATCHED_CN: "
        f"{idx + 1}/{len(results_cn)}"
    )

    results_cn.at[
        idx,
        "P2R_MATCHED_CN"
    ] = classify_sentiment(
        build_prompt_p2r_matched_cn(text)
    )

print("\nP2R_MATCHED_CN classification completed.")


# %%
# ============================================================
# 58. Check missing predictions
# ============================================================

print("Missing P2R_MATCHED_CN predictions:")

print(
    results_cn[
        "P2R_MATCHED_CN"
    ].isna().sum()
)

missing_matched_cn = results_cn[
    results_cn[
        "P2R_MATCHED_CN"
    ].isna()
].copy()

if len(missing_matched_cn) > 0:

    display(
        missing_matched_cn[
            [
                "text",
                "human_sentiment"
            ]
        ]
    )

else:

    print("No missing predictions.")


# %%
# ============================================================
# 59. Evaluate P2R_MATCHED_CN
# ============================================================

valid_matched_cn = results_cn[
    results_cn[
        "P2R_MATCHED_CN"
    ].notna()
].copy()

y_true_matched_cn = (
    valid_matched_cn[
        "human_sentiment"
    ]
    .astype(int)
)

y_pred_matched_cn = (
    valid_matched_cn[
        "P2R_MATCHED_CN"
    ]
    .astype(int)
)

matched_cn_metrics = {

    "prompt": "P2R_MATCHED_CN",

    "n": len(valid_matched_cn),

    "accuracy": accuracy_score(
        y_true_matched_cn,
        y_pred_matched_cn
    ),

    "macro_precision": precision_score(
        y_true_matched_cn,
        y_pred_matched_cn,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_true_matched_cn,
        y_pred_matched_cn,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_true_matched_cn,
        y_pred_matched_cn,
        average="macro",
        zero_division=0
    ),

    "cohen_kappa": cohen_kappa_score(
        y_true_matched_cn,
        y_pred_matched_cn
    )
}

matched_cn_metrics_df = pd.DataFrame(
    [matched_cn_metrics]
)

print("\nP2R_MATCHED_CN overall performance:")

display(
    matched_cn_metrics_df.round(4)
)


# %%
# ============================================================
# 60. Compare all Chinese prompts
# ============================================================

prompt_columns_cn_matched = [
    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN",
    "P2R2_CN",
    "P2R_MATCHED_CN"
]

evaluation_rows_cn_matched = []

for prompt_col in prompt_columns_cn_matched:

    valid = results_cn[
        results_cn[
            prompt_col
        ].notna()
    ].copy()

    y_true = (
        valid[
            "human_sentiment"
        ]
        .astype(int)
    )

    y_pred = (
        valid[
            prompt_col
        ]
        .astype(int)
    )

    evaluation_rows_cn_matched.append({

        "prompt": prompt_col,

        "n": len(valid),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "cohen_kappa": cohen_kappa_score(
            y_true,
            y_pred
        )
    })


evaluation_cn_matched_df = pd.DataFrame(
    evaluation_rows_cn_matched
)

print("\nAll Chinese prompt comparison:")

display(
    evaluation_cn_matched_df.round(4)
)


# %%
# ============================================================
# 61. Class-level performance
#     P2R_MATCHED_CN
# ============================================================

matched_cn_report = classification_report(
    y_true_matched_cn,
    y_pred_matched_cn,
    labels=[-1, 0, 1],
    target_names=[
        "Negative",
        "Neutral",
        "Positive"
    ],
    output_dict=True,
    zero_division=0
)

matched_cn_report_df = pd.DataFrame(
    matched_cn_report
).T

print(
    "\n"
    + "=" * 100
)

print(
    "CLASS-LEVEL PERFORMANCE: "
    "P2R_MATCHED_CN"
)

print(
    "=" * 100
)

display(
    matched_cn_report_df[
        [
            "precision",
            "recall",
            "f1-score",
            "support"
        ]
    ].round(4)
)


# %%
# ============================================================
# 62. Confusion matrix
# ============================================================

matched_cn_cm = confusion_matrix(
    y_true_matched_cn,
    y_pred_matched_cn,
    labels=[-1, 0, 1]
)

matched_cn_cm_df = pd.DataFrame(
    matched_cn_cm,
    index=[
        "Human Negative",
        "Human Neutral",
        "Human Positive"
    ],
    columns=[
        "Pred Negative",
        "Pred Neutral",
        "Pred Positive"
    ]
)

print(
    "\nP2R_MATCHED_CN confusion matrix:"
)

display(
    matched_cn_cm_df
)


# %%
# ============================================================
# 63. Compare label counts
# ============================================================

columns_to_check_cn_matched = [
    "human_sentiment",
    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN",
    "P2R2_CN",
    "P2R_MATCHED_CN"
]

distribution_cn_matched = {}

for col in columns_to_check_cn_matched:

    counts = (
        results_cn[col]
        .dropna()
        .astype(int)
        .value_counts()
        .reindex(
            [-1, 0, 1],
            fill_value=0
        )
    )

    distribution_cn_matched[
        col
    ] = counts


distribution_cn_matched_df = pd.DataFrame(
    distribution_cn_matched
)

distribution_cn_matched_df.index = [
    "Negative (-1)",
    "Neutral (0)",
    "Positive (1)"
]

print("\nLabel counts:")

display(
    distribution_cn_matched_df
)


# %%
# ============================================================
# 64. Compare label percentages
# ============================================================

distribution_pct_cn_matched_df = (
    distribution_cn_matched_df
    /
    distribution_cn_matched_df.sum(
        axis=0
    )
    *
    100
)

print("\nLabel percentages:")

display(
    distribution_pct_cn_matched_df.round(2)
)


# %%
# ============================================================
# 65. Error analysis
# ============================================================

matched_cn_errors = results_cn[
    results_cn[
        "P2R_MATCHED_CN"
    ].notna()
].copy()

matched_cn_errors = matched_cn_errors[
    matched_cn_errors[
        "P2R_MATCHED_CN"
    ].astype(int)
    !=
    matched_cn_errors[
        "human_sentiment"
    ].astype(int)
].copy()

print(
    "\n"
    + "=" * 100
)

print(
    "ERRORS FOR "
    "P2R_MATCHED_CN"
)

print(
    "=" * 100
)

print(
    "Number of errors:",
    len(matched_cn_errors)
)

display(
    matched_cn_errors[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_MATCHED_CN"
        ]
    ]
)


# %%
# ============================================================
# 66. Error transitions
# ============================================================

matched_cn_transition = (
    matched_cn_errors
    .assign(
        human_sentiment=lambda x:
            x[
                "human_sentiment"
            ].astype(int),

        P2R_MATCHED_CN=lambda x:
            x[
                "P2R_MATCHED_CN"
            ].astype(int)
    )
    .groupby(
        [
            "human_sentiment",
            "P2R_MATCHED_CN"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

print(
    "\nP2R_MATCHED_CN "
    "error transitions:"
)

display(
    matched_cn_transition
)


# %%
# ============================================================
# 67. Neutral -> Positive errors
# ============================================================

matched_neutral_to_positive = results_cn[
    results_cn[
        "P2R_MATCHED_CN"
    ].notna()
].copy()

matched_neutral_to_positive = (
    matched_neutral_to_positive[
        (
            matched_neutral_to_positive[
                "human_sentiment"
            ].astype(int) == 0
        )
        &
        (
            matched_neutral_to_positive[
                "P2R_MATCHED_CN"
            ].astype(int) == 1
        )
    ]
    .copy()
)

print(
    "\n"
    + "=" * 100
)

print(
    "P2R_MATCHED_CN: "
    "HUMAN NEUTRAL -> MODEL POSITIVE"
)

print(
    "=" * 100
)

print(
    "Number of Neutral -> Positive errors:",
    len(
        matched_neutral_to_positive
    )
)

display(
    matched_neutral_to_positive[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_MATCHED_CN"
        ]
    ]
)


# %%
# ============================================================
# 68. Positive -> Neutral errors
# ============================================================

matched_positive_to_neutral = results_cn[
    results_cn[
        "P2R_MATCHED_CN"
    ].notna()
].copy()

matched_positive_to_neutral = (
    matched_positive_to_neutral[
        (
            matched_positive_to_neutral[
                "human_sentiment"
            ].astype(int) == 1
        )
        &
        (
            matched_positive_to_neutral[
                "P2R_MATCHED_CN"
            ].astype(int) == 0
        )
    ]
    .copy()
)

print(
    "\n"
    + "=" * 100
)

print(
    "P2R_MATCHED_CN: "
    "HUMAN POSITIVE -> MODEL NEUTRAL"
)

print(
    "=" * 100
)

print(
    "Number of Positive -> Neutral errors:",
    len(
        matched_positive_to_neutral
    )
)

display(
    matched_positive_to_neutral[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_MATCHED_CN"
        ]
    ]
)


# %%
# ============================================================
# 69. Direct comparison:
#     P2_CN vs P2R_MATCHED_CN
# ============================================================

comparison_p2_matched_cn = results_cn[
    results_cn[
        "P2_CN"
    ].notna()
    &
    results_cn[
        "P2R_MATCHED_CN"
    ].notna()
].copy()

comparison_p2_matched_cn[
    "P2_correct"
] = (
    comparison_p2_matched_cn[
        "P2_CN"
    ].astype(int)
    ==
    comparison_p2_matched_cn[
        "human_sentiment"
    ].astype(int)
)

comparison_p2_matched_cn[
    "MATCHED_correct"
] = (
    comparison_p2_matched_cn[
        "P2R_MATCHED_CN"
    ].astype(int)
    ==
    comparison_p2_matched_cn[
        "human_sentiment"
    ].astype(int)
)


fixed_by_matched_cn = (
    comparison_p2_matched_cn[
        (
            ~comparison_p2_matched_cn[
                "P2_correct"
            ]
        )
        &
        (
            comparison_p2_matched_cn[
                "MATCHED_correct"
            ]
        )
    ]
    .copy()
)


worsened_by_matched_cn = (
    comparison_p2_matched_cn[
        (
            comparison_p2_matched_cn[
                "P2_correct"
            ]
        )
        &
        (
            ~comparison_p2_matched_cn[
                "MATCHED_correct"
            ]
        )
    ]
    .copy()
)


print(
    "\n"
    + "=" * 100
)

print(
    "CASES FIXED BY "
    "P2R_MATCHED_CN"
)

print(
    "=" * 100
)

print(
    "Number fixed:",
    len(
        fixed_by_matched_cn
    )
)

display(
    fixed_by_matched_cn[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_MATCHED_CN"
        ]
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "CASES MADE WORSE BY "
    "P2R_MATCHED_CN"
)

print(
    "=" * 100
)

print(
    "Number worsened:",
    len(
        worsened_by_matched_cn
    )
)

display(
    worsened_by_matched_cn[
        [
            "text",
            "human_sentiment",
            "P2_CN",
            "P2R_MATCHED_CN"
        ]
    ]
)


# %%
# ============================================================
# 70. P2 vs matched P2R summary
# ============================================================

p2_vs_matched_cn_summary = pd.DataFrame({

    "metric": [
        "Cases fixed by matched P2R",
        "Cases worsened by matched P2R",
        "Net improvement"
    ],

    "value": [
        len(
            fixed_by_matched_cn
        ),

        len(
            worsened_by_matched_cn
        ),

        len(
            fixed_by_matched_cn
        )
        -
        len(
            worsened_by_matched_cn
        )
    ]
})

print(
    "\nP2_CN vs "
    "P2R_MATCHED_CN summary:"
)

display(
    p2_vs_matched_cn_summary
)


# %%
# ============================================================
# 71. Save full matched-P2R analysis workbook
# ============================================================

FINAL_MATCHED_CN_PATH = (
    BASE_DIR /
    "xinhua_sentiment_prompt_validation_"
    "matched_P2R_full_analysis.xlsx"
)

with pd.ExcelWriter(
    FINAL_MATCHED_CN_PATH,
    engine="openpyxl"
) as writer:

    results_cn.to_excel(
        writer,
        sheet_name="article_results",
        index=False
    )

    evaluation_cn_matched_df.to_excel(
        writer,
        sheet_name="overall_metrics",
        index=False
    )

    matched_cn_report_df.to_excel(
        writer,
        sheet_name="matched_class_metrics"
    )

    matched_cn_cm_df.to_excel(
        writer,
        sheet_name="matched_confusion"
    )

    distribution_cn_matched_df.to_excel(
        writer,
        sheet_name="label_counts"
    )

    distribution_pct_cn_matched_df.to_excel(
        writer,
        sheet_name="label_percentages"
    )

    matched_cn_errors.to_excel(
        writer,
        sheet_name="matched_errors",
        index=False
    )

    matched_cn_transition.to_excel(
        writer,
        sheet_name="error_transitions",
        index=False
    )

    matched_neutral_to_positive.to_excel(
        writer,
        sheet_name="neutral_to_positive",
        index=False
    )

    matched_positive_to_neutral.to_excel(
        writer,
        sheet_name="positive_to_neutral",
        index=False
    )

    fixed_by_matched_cn.to_excel(
        writer,
        sheet_name="fixed_vs_P2",
        index=False
    )

    worsened_by_matched_cn.to_excel(
        writer,
        sheet_name="worsened_vs_P2",
        index=False
    )

    p2_vs_matched_cn_summary.to_excel(
        writer,
        sheet_name="P2_vs_matched_summary",
        index=False
    )


print(
    "\nFinal matched P2R-CN "
    "analysis workbook saved:"
)

print(
    FINAL_MATCHED_CN_PATH
)

Processing P2R_MATCHED_CN: 1/116
Processing P2R_MATCHED_CN: 2/116
Processing P2R_MATCHED_CN: 3/116
Processing P2R_MATCHED_CN: 4/116
Processing P2R_MATCHED_CN: 5/116
Processing P2R_MATCHED_CN: 6/116
Processing P2R_MATCHED_CN: 7/116
Processing P2R_MATCHED_CN: 8/116
Processing P2R_MATCHED_CN: 9/116
Processing P2R_MATCHED_CN: 10/116
Processing P2R_MATCHED_CN: 11/116
Processing P2R_MATCHED_CN: 12/116
Processing P2R_MATCHED_CN: 13/116
Processing P2R_MATCHED_CN: 14/116
Processing P2R_MATCHED_CN: 15/116
Processing P2R_MATCHED_CN: 16/116
Processing P2R_MATCHED_CN: 17/116
Processing P2R_MATCHED_CN: 18/116
Processing P2R_MATCHED_CN: 19/116
Processing P2R_MATCHED_CN: 20/116
Processing P2R_MATCHED_CN: 21/116
Processing P2R_MATCHED_CN: 22/116
Processing P2R_MATCHED_CN: 23/116
Processing P2R_MATCHED_CN: 24/116
Processing P2R_MATCHED_CN: 25/116
Processing P2R_MATCHED_CN: 26/116
Processing P2R_MATCHED_CN: 27/116
Processing P2R_MATCHED_CN: 28/116
Processing P2R_MATCHED_CN: 29/116
Processing P2R_MATCHED_

,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P2R_MATCHED_CN,116,0.8448,0.8756,0.5978,0.6726,0.4202



All Chinese prompt comparison:


,prompt,n,accuracy,macro_precision,macro_recall,macro_f1,cohen_kappa
0,P1_CN,116,0.8362,0.8291,0.6374,0.7025,0.4769
1,P2_CN,116,0.8362,0.8281,0.6483,0.7093,0.4932
2,P3_CN,116,0.7759,0.7857,0.6551,0.6860,0.4237
3,P2R_CN,116,0.7759,0.7857,0.6551,0.6860,0.4237
4,P2R2_CN,116,0.8362,0.8403,0.6050,0.6770,0.4208
5,P2R_MATCHED_CN,116,0.8448,0.8756,0.5978,0.6726,0.4202



CLASS-LEVEL PERFORMANCE: P2R_MATCHED_CN


...


CASES MADE WORSE BY P2R_MATCHED_CN
Number worsened: 6


,text,human_sentiment,P2_CN,P2R_MATCHED_CN
2,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,0,0,1
15,谷歌母公司“字母表”旗下的“出行新方式”（Ｗａｙｍｏ）公司９日宣布，将从下周开始在美国亚特兰...,0,0,1
27,在北美轿车市场萎缩的情况下，通用汽车不久前宣布在美国和加拿大大幅减员，同时削减滞销车型的生产...,0,0,1
54,3D扫描定制西服、无人驾驶汽车驶上公路、智能生产车间……在中国重庆举行的中国国际智能产业博览...,0,0,1
105,同时，新版清单依法规范重点领域准入，依据已出台的法律、行政法规、国务院决定等，对部分领域市场...,0,0,1
115,在墨西哥城世贸中心展览馆二层展厅，来自浙江、福建、广东等地的超过３００家企业参展，展出小至手...,0,0,1



P2_CN vs P2R_MATCHED_CN summary:


,metric,value
0,Cases fixed by matched P2R,7
1,Cases worsened by matched P2R,6
2,Net improvement,1



Final matched P2R-CN analysis workbook saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_sentiment_prompt_validation_matched_P2R_full_analysis.xlsx
